# Atlas — слепой эксперимент Навье—Стокса

Этот notebook запускает **штатный** `AdaptiveResearchKernelOwner` из release `0.15.28.0` через модуль `evaluation.navier_stokes_blind_experiment`. Он содержит четыре уровня: masked-term control, primitive-field discovery, operator-language invention и residual-driven hidden-term discovery.

Atlas не получает название governing equation, формулу или расшифровку масок до окончания execution receipts. Первые три уровня проверяют momentum-компоненты/closure; Level 4 проверяет persistent residual -> algebra-depth expansion -> hidden candidate -> sealed OOD на отдельном controlled reference world.

**Граница:** это benchmark восстановления структуры controlled reference world, а не доказательство существования/гладкости Навье—Стокса и не новый закон природы.


In [ ]:
from pathlib import Path
import json, sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "source").is_dir() and (p / "evaluation").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Atlas release root not found")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from source.lawspace.api import LawSpaceAPI
from evaluation.navier_stokes_blind_experiment import run, run_primitive_field, run_operator_language_invention, run_hidden_term_discovery

print("ROOT:", ROOT)
print("release:", LawSpaceAPI(ROOT).runtime.current_release_id())


## 1. Запуск полного blind benchmark

Следующая ячейка выполняет experiment одним вызовом и сохраняет полный JSON receipt. Не изменяйте `report` перед сохранением.


In [ ]:
report = run(ROOT)
print("status:", report["status"])
print("passed:", report["passed"], "/", report["total"])
print("digest:", report["digest"])


## 2. Ключевые результаты до физической интерпретации

Здесь показываются только masked coordinates и статистики Atlas.


In [ ]:
for lane in ("x_momentum", "y_momentum", "local_closure"):
    row = report["blind_results"][lane]
    print("\n", lane)
    print(" status:", row["status"])
    print(" activated:", row["activated_axes"])
    print(" initial NRMSE:", row["initial_holdout_nrmse"])
    print(" best:", row["best_expression"])
    print(" coeff:", row["best_coefficients_by_term"])
    print(" sealed NRMSE:", row["sealed_holdout"]["nrmse"])


## 3. Журнал исполнения

Журнал хранит последовательность freeze → Atlas execution → post-freeze decode и digest каждой стадии.


In [ ]:
for entry in report["run_journal"]:
    print(entry)


## 4. Post-freeze decode

Эта секция читается **после** того, как execution receipts уже сформированы. Она нужна только для проверки того, какую известную физическую структуру восстановил masked search.


In [ ]:
print(json.dumps(report["postfreeze_decoding"], ensure_ascii=False, indent=2))


## 5. Сохранение результата для передачи

Передайте созданный файл без ручного редактирования.


In [ ]:
OUT = ROOT / "reports" / "NAVIER_STOKES_BLIND_EXPERIMENT_CURRENT.json"
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("saved:", OUT)
print("bytes:", OUT.stat().st_size)


## Acceptance

Нормальный результат текущего control: `PASS_BLIND_CONTINUUM_BALANCE_RECOVERY`. Если статус другой, не подгоняйте параметры и не редактируйте данные. Сохраните JSON и terminal/notebook traceback — это и будет исследовательский результат.


# 6. Primitive-field blind discovery

Во втором уровне Atlas получает только обезличенные sampled fields и coordinate arrays. Производные, convective products, pressure-gradient coordinates и diffusion coordinates рождаются внутри Theory Compiler. `AXIS_BIRTH_CARDINALITY=ADAPTIVE`, поэтому один research cycle может активировать сразу несколько dormant axes.


In [ ]:
primitive_report = run_primitive_field(ROOT)
print('status:', primitive_report['status'])
print('passed:', primitive_report['passed'], '/', primitive_report['total'])
print('digest:', primitive_report['digest'])


## 7. Multi-axis birth и sealed OOD


In [ ]:
for lane in ('x_momentum', 'y_momentum'):
    row = primitive_report['blind_results'][lane]
    print('\n', lane)
    print(' born candidates:', row['born_candidate_axis_count'])
    print(' baseline:', row['baseline_axis'])
    print(' activated:', row['activated_axes'])
    print(' selected birth cardinality:', row['selected_birth_cardinality'])
    print(' discovery NRMSE:', row['discovery_holdout_nrmse'])
    print(' sealed NRMSE:', row['sealed_holdout']['nrmse'])
    print(' coefficients:', row['coefficients'])


## 8. Сохранение primitive-field receipt


In [ ]:
PRIMITIVE_OUT = ROOT / 'reports' / 'NAVIER_STOKES_PRIMITIVE_FIELD_CURRENT.json'
PRIMITIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
PRIMITIVE_OUT.write_text(json.dumps(primitive_report, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(PRIMITIVE_OUT)


## Primitive-field acceptance

Нормальный статус текущего control: `PASS_PRIMITIVE_FIELD_BLIND_OPERATOR_DISCOVERY`. Если он не получен, не редактируйте fixture/JSON вручную: передайте полный receipt для разбора.


# 9. Level 3 — рождение языка операторов

В этом режиме Atlas не получает заранее заданную differential grammar `D/D2`. Mathematical Invention рождает локальные translation-moment signatures из более слабых meta-primitives: локальный сдвиг, линейная суперпозиция, точечное умножение/обратная величина и размерностная типизация.


In [ ]:
language_report = run_operator_language_invention(ROOT)
print('status:', language_report['status'])
print('passed:', language_report['passed'], '/', language_report['total'])
print('digest:', language_report['digest'])


## 10. Что родил Atlas до post-freeze decode


In [ ]:
for lane in ('x_momentum', 'y_momentum'):
    row = language_report['blind_results'][lane]
    print('\n', lane)
    print('generated signatures:', row['generated_signature_count'])
    print('rank shells:', row['generated_rank_shells'])
    print('selected birth cardinality:', row['selected_birth_cardinality'])
    print('discovery NRMSE:', row['discovery_holdout_nrmse'])
    print('sealed NRMSE:', row['sealed_holdout']['nrmse'])


## 11. Сохранение Level-3 receipt


In [ ]:
LANGUAGE_OUT = ROOT / 'reports' / 'NAVIER_STOKES_OPERATOR_LANGUAGE_INVENTION_CURRENT.json'
LANGUAGE_OUT.parent.mkdir(parents=True, exist_ok=True)
LANGUAGE_OUT.write_text(json.dumps(language_report, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(LANGUAGE_OUT)


## Level-3 acceptance

Нормальный контрольный статус: `PASS_BLIND_OPERATOR_LANGUAGE_INVENTION`. Этот PASS означает механизм рождение языка + support discovery в контролируемом reference world; он не означает новую физику и не означает изобретение дифференциального исчисления «из ничего».


## Level 4 — persistent residual → language expansion → hidden candidate

The frozen baseline is intentionally incomplete. The hidden correction is known only to the reference-world builder and is decoded after the Atlas receipt is sealed.


In [ ]:
level4 = run_hidden_term_discovery(ROOT)
print(level4["status"], level4["passed"], "/", level4["total"])
print(json.dumps(level4["blind_result"], ensure_ascii=False, indent=2, sort_keys=True))


In [ ]:
level4_path = ROOT / "reports" / "HIDDEN_TERM_RESIDUAL_DISCOVERY_CURRENT.json"
level4_path.write_text(json.dumps(level4, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(level4_path)


## Level 4.1 — top-level provenance seal

После Level-4 fit/sealed PASS отдельно проверяется верхний `execution_receipt.atlas_claim`. Общий acceptance теперь включает `TOP_LEVEL_ATLAS_PROVENANCE_ACCEPTED`; отсутствие обязательных digest bindings переводит весь benchmark в FAIL.

In [ ]:
claim = level4["execution_receipt"]["atlas_claim"]
assert level4["passed"] == 19 and level4["total"] == 19
assert claim["atlas_native"] is True
assert claim["status"] == "ATLAS_NATIVE_PROVENANCE_ACCEPTED_NOT_SCIENTIFIC_PROMOTION"
assert all(claim["checks"].values())
print({"status": level4["status"], "passed": level4["passed"], "total": level4["total"], "top_level_atlas_native": claim["atlas_native"], "claim_status": claim["status"]})